# Notebook 03 — Confidence Threshold Tuning

This notebook answers: **"Is the 90% confidence threshold the right choice?"**

Key concepts:
- **Recall** = how many actual tumors did we catch? (higher = fewer missed tumors)
- **Precision** = of all the "tumor" predictions, how many were right? (higher = fewer false alarms)
- **Precision-Recall Tradeoff**: raising the confidence threshold increases precision but lowers recall (we make fewer predictions but they are more reliable).
- **Our priority**: Recall ≥ 95% first, then maximize precision. Missing a real tumor is worse than a false alarm.

The **Precision-Recall curve** shows how these two metrics trade off across all possible thresholds. We pick the threshold that gives Recall ≥ 95% with the highest possible precision.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', '../requirements.txt', '-q'])
print('✓ All dependencies installed.')

In [ ]:
import sys, yaml
sys.path.insert(0, '..')

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import precision_recall_curve, roc_curve, auc

from src.data.dataset import BrainTumorDataset
from src.data.augmentation import get_val_transform
from src.model.classifier import load_model

with open('../config.yaml') as f:
    CFG = yaml.safe_load(f)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = load_model('../models/classifier_best.pt', DEVICE)
print('Model loaded.')

In [ ]:
# Collect raw probability scores on the validation set
val_dataset = BrainTumorDataset(
    root_dir='../data/Training',
    transform=get_val_transform(),
    index_file='../data/splits/val_index.csv',
)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

all_probs, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        logits = model(images.to(DEVICE))
        probs  = F.softmax(logits, dim=1)[:, 1].cpu().numpy()  # P(tumor)
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)
print(f'Collected {len(all_probs)} validation predictions.')

## Precision-Recall Curve

In [ ]:
precision, recall, thresholds = precision_recall_curve(all_labels, all_probs)
pr_auc = auc(recall, precision)

# Find the threshold where Recall is just at or above 95%
recall_target = 0.95
valid_idx = np.where(recall[:-1] >= recall_target)[0]
if len(valid_idx) > 0:
    best_idx = valid_idx[np.argmax(precision[valid_idx])]
    optimal_threshold = thresholds[best_idx]
    print(f'Optimal threshold for Recall >= {recall_target}: {optimal_threshold:.3f}')
    print(f'  Recall at this threshold:    {recall[best_idx]:.4f}')
    print(f'  Precision at this threshold: {precision[best_idx]:.4f}')
else:
    optimal_threshold = 0.50
    print(f'WARNING: Could not achieve Recall >= {recall_target}. Model may need more training.')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(recall, precision, 'b-', linewidth=2, label=f'PR curve (AUC={pr_auc:.3f})')
if len(valid_idx) > 0:
    axes[0].scatter(recall[best_idx], precision[best_idx], color='red', s=100, zorder=5,
                   label=f'Optimal @ thresh={optimal_threshold:.2f}')
axes[0].axvline(recall_target, color='orange', linestyle='--', label=f'Recall target ({recall_target*100:.0f}%)')
axes[0].set_xlabel('Recall (Sensitivity)')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve')
axes[0].legend()
axes[0].set_xlim([0, 1]); axes[0].set_ylim([0, 1])
axes[0].grid(True, alpha=0.3)

# ROC curve
fpr, tpr, roc_thresholds = roc_curve(all_labels, all_probs)
roc_auc_val = auc(fpr, tpr)
axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC curve (AUC={roc_auc_val:.3f})')
axes[1].plot([0,1],[0,1], 'k--', label='Random classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate (Recall)')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/threshold_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'ROC AUC: {roc_auc_val:.4f}')

## Uncertainty Rate Analysis

How often does the model produce results below the 90% threshold? 
If this is >20% on the validation set, the model's confidence is too spread out.

In [ ]:
CONF_THRESHOLD = CFG['inference']['high_confidence_threshold']  # 0.90

# P(tumor) and P(no_tumor) = 1 - P(tumor) under binary assumption
p_tumor    = all_probs
p_no_tumor = 1 - all_probs
max_conf   = np.maximum(p_tumor, p_no_tumor)

uncertain_mask = max_conf < CONF_THRESHOLD
uncertainty_rate = uncertain_mask.mean() * 100

print(f'Confidence threshold: {CONF_THRESHOLD * 100:.0f}%')
print(f'Uncertainty rate (val set): {uncertainty_rate:.1f}%')
print(f'  --> Target: < 20%')
if uncertainty_rate > 20:
    print('  WARNING: Uncertainty rate too high. Consider lowering the threshold or more training.')
else:
    print('  PASS: Uncertainty rate within acceptable range.')

# Visualize max-confidence distribution
plt.figure(figsize=(9, 4))
plt.hist(max_conf, bins=40, color='steelblue', edgecolor='white')
plt.axvline(CONF_THRESHOLD, color='red', linestyle='--', label=f'Threshold ({CONF_THRESHOLD*100:.0f}%)')
plt.xlabel('Max(P(tumor), P(no_tumor))')
plt.ylabel('Count')
plt.title('Model Confidence Distribution on Validation Set')
plt.legend()
plt.tight_layout()
plt.savefig('../docs/confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()